# 09 — Siamese CNN + Transformer with FCGR k=7

## Obiettivo

Valutare se un encoder ibrido CNN + Transformer riesce a produrre
embedding più discriminativi rispetto alla CNN V3 utilizzata negli
esperimenti precedenti.

La CNN viene utilizzata per estrarre pattern locali dalla FCGR,
mentre il Transformer opera sulle feature spaziali prodotte dalla CNN
per modellare relazioni globali tra regioni differenti della matrice.

Pipeline:

FCGR k=7 (128×128)
→ CNN frontend
→ feature map 128×16×16
→ 256 spatial tokens
→ Transformer Encoder
→ global pooling
→ embedding 128D
→ L2 normalization
→ Siamese Euclidean Contrastive Learning

Per mantenere un confronto controllato con il notebook 08:

- stesso dataset Tumor + Healthy a 12 classi
- stesso downsampling a 2111 campioni/classe
- stessi random pairs 50/50
- stessa Euclidean Contrastive Loss
- margin = 1.25
- embedding dimension = 128
- stessa validation
- FCGR k=7

In [1]:
# ============================================================
# IMPORTS + CONFIGURATION
# ============================================================

from pathlib import Path

import random
import time
import copy

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import (
    Dataset,
    DataLoader
)

from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)


# ============================================================
# PROJECT PATHS
# ============================================================

CURRENT_DIR = Path.cwd().resolve()

PROJECT_ROOT = (
    CURRENT_DIR.parent
    if CURRENT_DIR.name == "notebooks"
    else CURRENT_DIR
)


PROCESSED_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)


ARTIFACTS_DIR = (
    PROJECT_ROOT
    / "artifacts"
    / "siamese_cnn_transformer_k7"
)

ARTIFACTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# FILES
# ============================================================

MANIFEST_PATH = (
    PROCESSED_DIR
    / "siamese_tumor_healthy_manifest.tsv"
)


CLASS_MAPPING_PATH = (
    PROCESSED_DIR
    / "siamese_tumor_healthy_class_mapping.tsv"
)


VAL_POOL_PATH = (
    PROCESSED_DIR
    / "siamese_val_pair_pool.tsv"
)


# ============================================================
# FCGR
# ============================================================

K = 7

FCGR_SIZE = 2 ** K


FCGR_PATH = (
    PROCESSED_DIR
    / "fcgr_cache"
    / "fcgr_k7.npy"
)


FCGR_INDEX_PATH = (
    PROCESSED_DIR
    / "fcgr_cache"
    / "fcgr_k7_index.tsv"
)


# ============================================================
# EXPERIMENT
# ============================================================

RANDOM_STATE = 42

N_CLASSES = 12

EMBEDDING_DIM = 128

EUCLIDEAN_MARGIN = 1.25


TRAIN_PAIRS_PER_EPOCH = 50_000

VAL_PAIRS = 10_000

POSITIVE_FRACTION = 0.50


# Proviamo prima 64.
# Se il Transformer usa troppa VRAM scendiamo a 32.
BATCH_SIZE = 64


LEARNING_RATE = 3e-4

WEIGHT_DECAY = 1e-4


# ============================================================
# TRANSFORMER
# ============================================================

TRANSFORMER_DIM = 128

TRANSFORMER_HEADS = 4

TRANSFORMER_LAYERS = 3

TRANSFORMER_FF_DIM = 512

TRANSFORMER_DROPOUT = 0.10


# ============================================================
# DEVICE
# ============================================================

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


AMP_ENABLED = (
    DEVICE.type == "cuda"
)


print("=" * 72)
print("CNN + TRANSFORMER SIAMESE CONFIGURATION")
print("=" * 72)

print("Project:", PROJECT_ROOT)
print("Device:", DEVICE)

if DEVICE.type == "cuda":
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

print()

print(
    "FCGR:",
    f"k={K} → {FCGR_SIZE}x{FCGR_SIZE}"
)

print(
    "Embedding dim:",
    EMBEDDING_DIM
)

print(
    "Transformer dim:",
    TRANSFORMER_DIM
)

print(
    "Heads:",
    TRANSFORMER_HEADS
)

print(
    "Layers:",
    TRANSFORMER_LAYERS
)

print(
    "FF dimension:",
    TRANSFORMER_FF_DIM
)

print(
    "Batch size:",
    BATCH_SIZE
)

CNN + TRANSFORMER SIAMESE CONFIGURATION
Project: D:\Daria\Desktop\eccdna_fcgr_siamese
Device: cuda
GPU: NVIDIA GeForce RTX 3060 Laptop GPU

FCGR: k=7 → 128x128
Embedding dim: 128
Transformer dim: 128
Heads: 4
Layers: 3
FF dimension: 512
Batch size: 64


In [2]:
# ============================================================
# REPRODUCIBILITY
# ============================================================

def set_seed(seed):

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(
    RANDOM_STATE
)


if DEVICE.type == "cuda":

    torch.backends.cudnn.benchmark = True

    torch.set_float32_matmul_precision(
        "high"
    )


print(
    "Seed:",
    RANDOM_STATE
)

Seed: 42


In [3]:
# ============================================================
# LOAD MANIFEST
# ============================================================

metadata = pd.read_csv(
    MANIFEST_PATH,
    sep="\t",
    dtype={"id": str}
)


metadata["class_id"] = (
    metadata["class_id"]
    .astype(int)
)


class_mapping = pd.read_csv(
    CLASS_MAPPING_PATH,
    sep="\t"
)


print("=" * 72)
print("TUMOR + HEALTHY DATASET")
print("=" * 72)

print(
    "Samples:",
    len(metadata)
)

print(
    "Classes:",
    metadata["class_id"].nunique()
)

display(
    class_mapping
)

TUMOR + HEALTHY DATASET
Samples: 625937
Classes: 12


,original_class_id,disease_clean,disease_group,class_id
0,0,gastric cancer,cancer,0
1,1,healthy,healthy,1
2,2,ovarian cancer,cancer,2
3,3,prostate cancer,cancer,3
4,4,colorectal cancer,cancer,4
5,5,lymphoma,cancer,5
6,7,cervical adenocarcinoma,cancer,6
7,8,leukemia,cancer,7
8,11,hypopharyngeal squamous cell carcinoma,cancer,8
9,12,glioblastoma cancer,cancer,9


In [4]:
# ============================================================
# TRAIN SPLIT
# ============================================================

full_train_metadata = (
    metadata[
        metadata["split_cluster"]
        ==
        "train"
    ]
    .copy()
    .reset_index(drop=True)
)


train_counts = (
    full_train_metadata[
        "class_id"
    ]
    .value_counts()
    .sort_index()
)


MIN_CLASS_SIZE = int(
    train_counts.min()
)


balanced_parts = []


for class_id in sorted(
    full_train_metadata[
        "class_id"
    ].unique()
):

    class_df = (
        full_train_metadata[
            full_train_metadata["class_id"]
            ==
            class_id
        ]
    )


    sampled = class_df.sample(
        n=MIN_CLASS_SIZE,
        replace=False,
        random_state=(
            RANDOM_STATE
            +
            int(class_id)
        )
    )


    balanced_parts.append(
        sampled
    )


train_metadata = (
    pd.concat(
        balanced_parts,
        ignore_index=True
    )
    .reset_index(drop=True)
)


print("=" * 72)
print("BALANCED TRAIN")
print("=" * 72)

print(
    "Full train:",
    len(full_train_metadata)
)

print(
    "Samples/class:",
    MIN_CLASS_SIZE
)

print(
    "Balanced train:",
    len(train_metadata)
)

print(
    "Classes:",
    train_metadata[
        "class_id"
    ].nunique()
)

display(
    train_metadata[
        "class_id"
    ]
    .value_counts()
    .sort_index()
    .rename("samples")
    .to_frame()
)

BALANCED TRAIN
Full train: 96167
Samples/class: 2111
Balanced train: 25332
Classes: 12


,samples
class_id,
0,2111
1,2111
2,2111
3,2111
4,2111
5,2111
6,2111
7,2111
8,2111


In [5]:
# ============================================================
# VALIDATION POOL
# ============================================================

old_to_new = dict(
    zip(
        class_mapping[
            "original_class_id"
        ].astype(int),

        class_mapping[
            "class_id"
        ].astype(int)
    )
)


included_original_ids = set(
    old_to_new.keys()
)


val_original = pd.read_csv(
    VAL_POOL_PATH,
    sep="\t",
    dtype={"id": str}
)


val_original["class_id"] = (
    val_original["class_id"]
    .astype(int)
)


val_metadata = (
    val_original[
        val_original[
            "class_id"
        ].isin(
            included_original_ids
        )
    ]
    .copy()
    .reset_index(drop=True)
)


val_metadata[
    "original_class_id"
] = val_metadata[
    "class_id"
]


val_metadata[
    "class_id"
] = (
    val_metadata[
        "original_class_id"
    ]
    .map(
        old_to_new
    )
    .astype(int)
)


print("=" * 72)
print("VALIDATION")
print("=" * 72)

print(
    "Samples:",
    len(val_metadata)
)

print(
    "Classes:",
    val_metadata[
        "class_id"
    ].nunique()
)

display(
    val_metadata[
        "class_id"
    ]
    .value_counts()
    .sort_index()
    .rename("samples")
    .to_frame()
)

VALIDATION
Samples: 9753
Classes: 12


,samples
class_id,
0,1000
1,1000
2,1000
3,1000
4,1000
5,1000
6,1000
7,1000
8,585


In [6]:
# ============================================================
# LOAD FCGR k=7
# ============================================================

fcgr_memmap = np.load(
    FCGR_PATH,
    mmap_mode="r"
)


fcgr_index = pd.read_csv(
    FCGR_INDEX_PATH,
    sep="\t",
    dtype={"id": str}
)


id_to_fcgr_row = dict(
    zip(
        fcgr_index["id"],
        fcgr_index["fcgr_row"]
    )
)


missing_train = (
    ~train_metadata["id"]
    .isin(id_to_fcgr_row)
).sum()


missing_val = (
    ~val_metadata["id"]
    .isin(id_to_fcgr_row)
).sum()


print("=" * 72)
print("FCGR k=7")
print("=" * 72)

print(
    "Shape:",
    fcgr_memmap.shape
)

print(
    "dtype:",
    fcgr_memmap.dtype
)

print(
    "Missing train:",
    missing_train
)

print(
    "Missing val:",
    missing_val
)


assert fcgr_memmap.shape[1:] == (
    128,
    128
)

assert missing_train == 0

assert missing_val == 0

FCGR k=7
Shape: (150272, 128, 128)
dtype: float32
Missing train: 0
Missing val: 0


In [7]:
# ============================================================
# RANDOM SIAMESE PAIR DATASET
# ============================================================

class RandomSiamesePairDataset(Dataset):

    def __init__(
        self,
        metadata,
        fcgr_memmap,
        id_to_row,
        n_pairs,
        positive_fraction=0.50,
        seed=42,
        dynamic=True
    ):

        self.metadata = (
            metadata
            .copy()
            .reset_index(drop=True)
        )

        self.fcgr_memmap = fcgr_memmap

        self.n_pairs = int(
            n_pairs
        )

        self.positive_fraction = float(
            positive_fraction
        )

        self.seed = int(
            seed
        )

        self.dynamic = bool(
            dynamic
        )


        self.rows = (
            self.metadata["id"]
            .astype(str)
            .map(id_to_row)
            .to_numpy(dtype=np.int64)
        )


        self.labels = (
            self.metadata["class_id"]
            .to_numpy(dtype=np.int64)
        )


        self.classes = np.array(
            sorted(
                np.unique(
                    self.labels
                )
            ),
            dtype=np.int64
        )


        self.class_to_indices = {

            int(c):
                np.where(
                    self.labels
                    ==
                    c
                )[0]

            for c in self.classes
        }


        self._generate_pairs(
            self.seed
        )


    def _generate_pairs(
        self,
        seed
    ):

        rng = np.random.default_rng(
            seed
        )


        n_positive = int(
            round(
                self.n_pairs
                *
                self.positive_fraction
            )
        )


        targets = np.zeros(
            self.n_pairs,
            dtype=np.float32
        )


        targets[
            :n_positive
        ] = 1.0


        rng.shuffle(
            targets
        )


        anchors = rng.integers(
            0,
            len(self.labels),
            size=self.n_pairs
        )


        partners = np.empty(
            self.n_pairs,
            dtype=np.int64
        )


        for i in range(
            self.n_pairs
        ):

            anchor_idx = int(
                anchors[i]
            )


            anchor_class = int(
                self.labels[
                    anchor_idx
                ]
            )


            if targets[i] == 1.0:

                candidates = (
                    self.class_to_indices[
                        anchor_class
                    ]
                )


                partner_idx = (
                    anchor_idx
                )


                while (
                    partner_idx
                    ==
                    anchor_idx
                ):

                    partner_idx = int(
                        rng.choice(
                            candidates
                        )
                    )


            else:

                negative_classes = (
                    self.classes[
                        self.classes
                        !=
                        anchor_class
                    ]
                )


                negative_class = int(
                    rng.choice(
                        negative_classes
                    )
                )


                partner_idx = int(
                    rng.choice(
                        self.class_to_indices[
                            negative_class
                        ]
                    )
                )


            partners[i] = (
                partner_idx
            )


        self.row1 = (
            self.rows[
                anchors
            ]
        )

        self.row2 = (
            self.rows[
                partners
            ]
        )

        self.targets = (
            targets
        )


    def set_epoch(
        self,
        epoch
    ):

        if self.dynamic:

            self._generate_pairs(
                self.seed
                +
                int(epoch)
                *
                100_003
            )


    def __len__(
        self
    ):

        return self.n_pairs


    def __getitem__(
        self,
        index
    ):

        x1 = np.array(
            self.fcgr_memmap[
                int(
                    self.row1[index]
                )
            ],
            dtype=np.float32,
            copy=True
        )


        x2 = np.array(
            self.fcgr_memmap[
                int(
                    self.row2[index]
                )
            ],
            dtype=np.float32,
            copy=True
        )


        return {

            "x1":
                torch.from_numpy(
                    x1
                ).unsqueeze(0),

            "x2":
                torch.from_numpy(
                    x2
                ).unsqueeze(0),

            "target":
                torch.tensor(
                    self.targets[index],
                    dtype=torch.float32
                )
        }

In [8]:
# ============================================================
# PAIR DATASETS
# ============================================================

train_pair_dataset = RandomSiamesePairDataset(
    metadata=train_metadata,
    fcgr_memmap=fcgr_memmap,
    id_to_row=id_to_fcgr_row,
    n_pairs=TRAIN_PAIRS_PER_EPOCH,
    positive_fraction=POSITIVE_FRACTION,
    seed=RANDOM_STATE,
    dynamic=True
)


val_pair_dataset = RandomSiamesePairDataset(
    metadata=val_metadata,
    fcgr_memmap=fcgr_memmap,
    id_to_row=id_to_fcgr_row,
    n_pairs=VAL_PAIRS,
    positive_fraction=0.50,
    seed=RANDOM_STATE + 50_000,
    dynamic=False
)


train_pair_loader = DataLoader(
    train_pair_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=AMP_ENABLED
)


val_pair_loader = DataLoader(
    val_pair_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=AMP_ENABLED
)


print("=" * 72)
print("PAIR DATASETS")
print("=" * 72)

print(
    "Train pairs:",
    len(train_pair_dataset)
)

print(
    "Positive:",
    train_pair_dataset.targets.mean()
)

print(
    "Val pairs:",
    len(val_pair_dataset)
)

print(
    "Val positive:",
    val_pair_dataset.targets.mean()
)

print(
    "Train batches:",
    len(train_pair_loader)
)

PAIR DATASETS
Train pairs: 50000
Positive: 0.5
Val pairs: 10000
Val positive: 0.5
Train batches: 782


In [9]:
# ============================================================
# CNN + TRANSFORMER ENCODER
# ============================================================

class CNNTransformerEncoder(nn.Module):

    def __init__(
        self,
        embedding_dim=128,
        transformer_dim=128,
        n_heads=4,
        n_layers=3,
        ff_dim=512,
        dropout=0.10
    ):

        super().__init__()


        # ====================================================
        # CNN FRONTEND
        # ====================================================

        self.cnn = nn.Sequential(

            # 128x128
            nn.Conv2d(
                1,
                32,
                kernel_size=3,
                padding=1,
                bias=False
            ),

            nn.GroupNorm(
                8,
                32
            ),

            nn.GELU(),

            nn.MaxPool2d(2),


            # 64x64
            nn.Conv2d(
                32,
                64,
                kernel_size=3,
                padding=1,
                bias=False
            ),

            nn.GroupNorm(
                8,
                64
            ),

            nn.GELU(),

            nn.MaxPool2d(2),


            # 32x32
            nn.Conv2d(
                64,
                transformer_dim,
                kernel_size=3,
                padding=1,
                bias=False
            ),

            nn.GroupNorm(
                8,
                transformer_dim
            ),

            nn.GELU(),

            nn.MaxPool2d(2)

            # Output:
            # [B, 128, 16, 16]
        )


        # ====================================================
        # POSITION EMBEDDING
        # ====================================================

        self.n_tokens = (
            16
            *
            16
        )


        self.pos_embedding = nn.Parameter(
            torch.zeros(
                1,
                self.n_tokens,
                transformer_dim
            )
        )


        nn.init.trunc_normal_(
            self.pos_embedding,
            std=0.02
        )


        # ====================================================
        # TRANSFORMER
        # ====================================================

        transformer_layer = (
            nn.TransformerEncoderLayer(

                d_model=transformer_dim,

                nhead=n_heads,

                dim_feedforward=ff_dim,

                dropout=dropout,

                activation="gelu",

                batch_first=True,

                norm_first=True
            )
        )


        self.transformer = (
            nn.TransformerEncoder(
                transformer_layer,
                num_layers=n_layers
            )
        )


        self.final_norm = nn.LayerNorm(
            transformer_dim
        )


        # ====================================================
        # EMBEDDING HEAD
        # ====================================================

        self.embedding_head = nn.Sequential(

            nn.Linear(
                transformer_dim,
                256
            ),

            nn.GELU(),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                256,
                embedding_dim
            )
        )


    def forward(
        self,
        x
    ):

        # ----------------------------------------------------
        # CNN
        # ----------------------------------------------------

        x = self.cnn(
            x
        )

        # [B, C, 16, 16]


        batch_size = (
            x.shape[0]
        )


        # ----------------------------------------------------
        # FEATURE MAP → TOKENS
        # ----------------------------------------------------

        x = x.flatten(
            2
        )

        # [B, C, 256]


        x = x.transpose(
            1,
            2
        )

        # [B, 256, C]


        # ----------------------------------------------------
        # POSITIONAL INFORMATION
        # ----------------------------------------------------

        x = (
            x
            +
            self.pos_embedding
        )


        # ----------------------------------------------------
        # GLOBAL SELF-ATTENTION
        # ----------------------------------------------------

        x = self.transformer(
            x
        )


        x = self.final_norm(
            x
        )


        # ----------------------------------------------------
        # GLOBAL MEAN POOL
        # ----------------------------------------------------

        x = x.mean(
            dim=1
        )

        # [B, 128]


        # ----------------------------------------------------
        # FINAL EMBEDDING
        # ----------------------------------------------------

        z = self.embedding_head(
            x
        )


        return F.normalize(
            z,
            p=2,
            dim=1,
            eps=1e-8
        )

In [10]:
# ============================================================
# SIAMESE CNN + TRANSFORMER
# ============================================================

class SiameseCNNTransformer(nn.Module):

    def __init__(
        self
    ):

        super().__init__()


        self.encoder = (
            CNNTransformerEncoder(

                embedding_dim=
                    EMBEDDING_DIM,

                transformer_dim=
                    TRANSFORMER_DIM,

                n_heads=
                    TRANSFORMER_HEADS,

                n_layers=
                    TRANSFORMER_LAYERS,

                ff_dim=
                    TRANSFORMER_FF_DIM,

                dropout=
                    TRANSFORMER_DROPOUT
            )
        )


    def forward(
        self,
        x1,
        x2
    ):

        batch_size = (
            x1.shape[0]
        )


        x = torch.cat(
            [
                x1,
                x2
            ],
            dim=0
        )


        z = self.encoder(
            x
        )


        z1 = z[
            :batch_size
        ]


        z2 = z[
            batch_size:
        ]


        return (
            z1,
            z2
        )


model = SiameseCNNTransformer().to(
    DEVICE
)


n_parameters = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)


print("=" * 72)
print("SIAMESE CNN + TRANSFORMER")
print("=" * 72)

print(
    "Trainable parameters:",
    f"{n_parameters:,}"
)

C:\Users\simon\AppData\Local\Temp\ipykernel_25936\309768529.py:138: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  nn.TransformerEncoder(


SIAMESE CNN + TRANSFORMER
Trainable parameters: 786,656


In [11]:
# ============================================================
# ENCODER SANITY CHECK
# ============================================================

sample_rows = (
    train_metadata[
        "id"
    ]
    .iloc[:8]
    .map(
        id_to_fcgr_row
    )
    .to_numpy(
        dtype=np.int64
    )
)


x = np.stack(
    [
        np.array(
            fcgr_memmap[
                int(row)
            ],
            dtype=np.float32,
            copy=True
        )
        for row in sample_rows
    ]
)


x = torch.from_numpy(
    x
).unsqueeze(1).to(
    DEVICE
)


model.eval()


with torch.no_grad():

    cnn_features = (
        model.encoder.cnn(
            x
        )
    )


    z = model.encoder(
        x
    )


print("=" * 72)
print("CNN + TRANSFORMER SANITY CHECK")
print("=" * 72)

print(
    "Input:",
    x.shape
)

print(
    "CNN feature map:",
    cnn_features.shape
)

print(
    "Spatial tokens:",
    cnn_features.shape[-2]
    *
    cnn_features.shape[-1]
)

print(
    "Embedding:",
    z.shape
)

print(
    "Mean embedding norm:",
    z.norm(
        dim=1
    ).mean().item()
)

print(
    "Finite:",
    torch.isfinite(
        z
    ).all().item()
)

CNN + TRANSFORMER SANITY CHECK
Input: torch.Size([8, 1, 128, 128])
CNN feature map: torch.Size([8, 128, 16, 16])
Spatial tokens: 256
Embedding: torch.Size([8, 128])
Mean embedding norm: 1.0
Finite: True


In [12]:
# ============================================================
# EUCLIDEAN CONTRASTIVE LOSS
# ============================================================

class EuclideanContrastiveLoss(nn.Module):

    def __init__(
        self,
        margin=1.25
    ):

        super().__init__()

        self.margin = float(
            margin
        )


    def forward(
        self,
        z1,
        z2,
        target
    ):

        z1 = z1.float()

        z2 = z2.float()

        target = target.float()


        distances = (
            torch.linalg.vector_norm(
                z1 - z2,
                ord=2,
                dim=1
            )
        )


        positive_loss = (
            target
            *
            distances.pow(2)
        )


        negative_loss = (
            (1.0 - target)
            *
            F.relu(
                self.margin
                -
                distances
            ).pow(2)
        )


        loss = (
            positive_loss
            +
            negative_loss
        ).mean()


        return (
            loss,
            distances
        )


criterion = EuclideanContrastiveLoss(
    margin=EUCLIDEAN_MARGIN
)

In [13]:
# ============================================================
# PAIRWISE EVALUATION
# ============================================================

def evaluate_pairwise(
    model,
    loader,
    criterion
):

    model.eval()


    total_loss = 0.0

    total_samples = 0


    all_targets = []

    all_distances = []


    with torch.no_grad():

        for batch in loader:

            x1 = batch["x1"].to(
                DEVICE,
                non_blocking=True
            )

            x2 = batch["x2"].to(
                DEVICE,
                non_blocking=True
            )

            target = batch["target"].to(
                DEVICE,
                non_blocking=True
            )


            with torch.autocast(
                device_type=DEVICE.type,
                dtype=(
                    torch.float16
                    if DEVICE.type == "cuda"
                    else torch.bfloat16
                ),
                enabled=AMP_ENABLED
            ):

                z1, z2 = model(
                    x1,
                    x2
                )


            loss, distances = criterion(
                z1,
                z2,
                target
            )


            n = target.shape[0]


            total_loss += (
                loss.item()
                *
                n
            )

            total_samples += n


            all_targets.append(
                target.cpu()
                .numpy()
            )

            all_distances.append(
                distances.cpu()
                .numpy()
            )


    targets = np.concatenate(
        all_targets
    )


    distances = np.concatenate(
        all_distances
    )


    positive = distances[
        targets == 1
    ]


    negative = distances[
        targets == 0
    ]


    d_pos = float(
        positive.mean()
    )


    d_neg = float(
        negative.mean()
    )


    gap = (
        d_neg
        -
        d_pos
    )


    pooled_variance = (
        0.5
        *
        (
            positive.var()
            +
            negative.var()
        )
    )


    d_prime = float(
        gap
        /
        np.sqrt(
            pooled_variance
            +
            1e-12
        )
    )


    auc = float(
        roc_auc_score(
            targets,
            -distances
        )
    )


    return {

        "loss":
            total_loss
            /
            total_samples,

        "auc":
            auc,

        "d_pos":
            d_pos,

        "d_neg":
            d_neg,

        "gap":
            gap,

        "d_prime":
            d_prime
    }

In [14]:
# ============================================================
# TRAIN ONE EPOCH
# ============================================================

def train_one_epoch(
    model,
    loader,
    dataset,
    criterion,
    optimizer,
    scaler,
    epoch
):

    model.train()


    dataset.set_epoch(
        epoch
    )


    total_loss = 0.0

    total_samples = 0


    all_targets = []

    all_distances = []


    start_time = (
        time.perf_counter()
    )


    for batch in loader:

        x1 = batch["x1"].to(
            DEVICE,
            non_blocking=True
        )

        x2 = batch["x2"].to(
            DEVICE,
            non_blocking=True
        )

        target = batch["target"].to(
            DEVICE,
            non_blocking=True
        )


        optimizer.zero_grad(
            set_to_none=True
        )


        with torch.autocast(
            device_type=DEVICE.type,
            dtype=(
                torch.float16
                if DEVICE.type == "cuda"
                else torch.bfloat16
            ),
            enabled=AMP_ENABLED
        ):

            z1, z2 = model(
                x1,
                x2
            )


        loss, distances = criterion(
            z1,
            z2,
            target
        )


        if AMP_ENABLED:

            scaler.scale(
                loss
            ).backward()


            # Transformer:
            # evitiamo gradienti estremi.
            scaler.unscale_(
                optimizer
            )


            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0
            )


            scaler.step(
                optimizer
            )

            scaler.update()


        else:

            loss.backward()


            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0
            )


            optimizer.step()


        n = target.shape[0]


        total_loss += (
            loss.detach().item()
            *
            n
        )

        total_samples += n


        all_targets.append(
            target.detach()
            .cpu()
            .numpy()
        )


        all_distances.append(
            distances.detach()
            .cpu()
            .numpy()
        )


    targets = np.concatenate(
        all_targets
    )


    distances = np.concatenate(
        all_distances
    )


    positive = distances[
        targets == 1
    ]


    negative = distances[
        targets == 0
    ]


    return {

        "loss":
            total_loss
            /
            total_samples,

        "auc":
            float(
                roc_auc_score(
                    targets,
                    -distances
                )
            ),

        "d_pos":
            float(
                positive.mean()
            ),

        "d_neg":
            float(
                negative.mean()
            ),

        "gap":
            float(
                negative.mean()
                -
                positive.mean()
            ),

        "seconds":
            (
                time.perf_counter()
                -
                start_time
            )
    }

In [15]:
# ============================================================
# CNN + TRANSFORMER GPU BENCHMARK
# ============================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)


scaler = torch.amp.GradScaler(
    "cuda",
    enabled=AMP_ENABLED
)


batch = next(
    iter(train_pair_loader)
)


x1 = batch["x1"].to(
    DEVICE
)

x2 = batch["x2"].to(
    DEVICE
)

target = batch["target"].to(
    DEVICE
)


if DEVICE.type == "cuda":

    torch.cuda.empty_cache()

    torch.cuda.reset_peak_memory_stats()


# ============================================================
# WARMUP
# ============================================================

for _ in range(2):

    optimizer.zero_grad(
        set_to_none=True
    )


    with torch.autocast(
        device_type=DEVICE.type,
        dtype=torch.float16,
        enabled=AMP_ENABLED
    ):

        z1, z2 = model(
            x1,
            x2
        )


    loss, _ = criterion(
        z1,
        z2,
        target
    )


    scaler.scale(
        loss
    ).backward()


    scaler.unscale_(
        optimizer
    )


    torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        1.0
    )


    scaler.step(
        optimizer
    )

    scaler.update()


if DEVICE.type == "cuda":
    torch.cuda.synchronize()


# ============================================================
# BENCHMARK
# ============================================================

N_BENCHMARK_STEPS = 5


start = time.perf_counter()


for _ in range(
    N_BENCHMARK_STEPS
):

    optimizer.zero_grad(
        set_to_none=True
    )


    with torch.autocast(
        device_type=DEVICE.type,
        dtype=torch.float16,
        enabled=AMP_ENABLED
    ):

        z1, z2 = model(
            x1,
            x2
        )


    loss, _ = criterion(
        z1,
        z2,
        target
    )


    scaler.scale(
        loss
    ).backward()


    scaler.unscale_(
        optimizer
    )


    torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        1.0
    )


    scaler.step(
        optimizer
    )

    scaler.update()


if DEVICE.type == "cuda":
    torch.cuda.synchronize()


elapsed = (
    time.perf_counter()
    -
    start
)


seconds_per_batch = (
    elapsed
    /
    N_BENCHMARK_STEPS
)


estimated_epoch_seconds = (
    seconds_per_batch
    *
    len(train_pair_loader)
)


if DEVICE.type == "cuda":

    peak_memory_gb = (
        torch.cuda.max_memory_allocated()
        /
        1024**3
    )

else:

    peak_memory_gb = (
        float("nan")
    )


print("=" * 72)
print("CNN + TRANSFORMER BENCHMARK")
print("=" * 72)

print(
    "Batch size:",
    BATCH_SIZE
)

print(
    "Seconds/batch:",
    f"{seconds_per_batch:.3f}"
)

print(
    "Estimated epoch:",
    f"{estimated_epoch_seconds:.1f}s"
)

print(
    "Estimated epoch:",
    f"{estimated_epoch_seconds / 60:.2f} min"
)

print(
    "Peak GPU memory:",
    f"{peak_memory_gb:.2f} GB"
)

CNN + TRANSFORMER BENCHMARK
Batch size: 64
Seconds/batch: 0.094
Estimated epoch: 73.2s
Estimated epoch: 1.22 min
Peak GPU memory: 2.17 GB


In [16]:
# ============================================================
# RESET MODEL AFTER BENCHMARK
# ============================================================

set_seed(
    RANDOM_STATE
)


model = SiameseCNNTransformer().to(
    DEVICE
)


optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)


scaler = torch.amp.GradScaler(
    "cuda",
    enabled=AMP_ENABLED
)


print(
    "Model reset after benchmark: OK"
)

Model reset after benchmark: OK


C:\Users\simon\AppData\Local\Temp\ipykernel_25936\309768529.py:138: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  nn.TransformerEncoder(


In [17]:
# ============================================================
# CNN + TRANSFORMER SMOKE TEST
# ============================================================

SMOKE_EPOCHS = 3


smoke_initial_state = copy.deepcopy(
    model.state_dict()
)


print("=" * 118)
print("SIAMESE CNN + TRANSFORMER k=7 — SMOKE TEST")
print("=" * 118)


for epoch in range(
    1,
    SMOKE_EPOCHS + 1
):

    train_metrics = train_one_epoch(
        model=model,
        loader=train_pair_loader,
        dataset=train_pair_dataset,
        criterion=criterion,
        optimizer=optimizer,
        scaler=scaler,
        epoch=epoch
    )


    val_metrics = evaluate_pairwise(
        model=model,
        loader=val_pair_loader,
        criterion=criterion
    )


    print(
        f"Epoch {epoch:02d}/{SMOKE_EPOCHS}"

        f" | train loss "
        f"{train_metrics['loss']:.4f}"

        f" | train AUC "
        f"{train_metrics['auc']:.4f}"

        f" | val loss "
        f"{val_metrics['loss']:.4f}"

        f" | val AUC "
        f"{val_metrics['auc']:.4f}"

        f" | d+ "
        f"{val_metrics['d_pos']:.4f}"

        f" | d- "
        f"{val_metrics['d_neg']:.4f}"

        f" | gap "
        f"{val_metrics['gap']:.4f}"

        f" | d' "
        f"{val_metrics['d_prime']:.4f}"

        f" | "
        f"{train_metrics['seconds']:.1f}s"
    )

SIAMESE CNN + TRANSFORMER k=7 — SMOKE TEST
Epoch 01/3 | train loss 0.3829 | train AUC 0.5975 | val loss 0.4411 | val AUC 0.6034 | d+ 0.4278 | d- 0.5410 | gap 0.1132 | d' 0.3612 | 186.2s
Epoch 02/3 | train loss 0.3767 | train AUC 0.6123 | val loss 0.4072 | val AUC 0.6181 | d+ 0.4917 | d- 0.6126 | gap 0.1210 | d' 0.4193 | 136.7s
Epoch 03/3 | train loss 0.3719 | train AUC 0.6289 | val loss 0.3963 | val AUC 0.6304 | d+ 0.4402 | d- 0.5475 | gap 0.1073 | d' 0.4674 | 140.0s


In [18]:
# ============================================================
# EXTENDED SMOKE TEST — EPOCHS 4-6
# ============================================================

EXTRA_SMOKE_EPOCHS = 3

print("=" * 118)
print("SIAMESE CNN + TRANSFORMER k=7 — EXTENDED SMOKE TEST")
print("=" * 118)


extended_results = []


for epoch in range(
    SMOKE_EPOCHS + 1,
    SMOKE_EPOCHS + EXTRA_SMOKE_EPOCHS + 1
):

    train_metrics = train_one_epoch(
        model=model,
        loader=train_pair_loader,
        dataset=train_pair_dataset,
        criterion=criterion,
        optimizer=optimizer,
        scaler=scaler,
        epoch=epoch
    )


    val_metrics = evaluate_pairwise(
        model=model,
        loader=val_pair_loader,
        criterion=criterion
    )


    extended_results.append(
        {
            "epoch": epoch,
            "train_auc": train_metrics["auc"],
            "val_auc": val_metrics["auc"],
            "val_gap": val_metrics["gap"],
            "val_d_prime": val_metrics["d_prime"]
        }
    )


    print(
        f"Epoch {epoch:02d}/6"

        f" | train loss "
        f"{train_metrics['loss']:.4f}"

        f" | train AUC "
        f"{train_metrics['auc']:.4f}"

        f" | val loss "
        f"{val_metrics['loss']:.4f}"

        f" | val AUC "
        f"{val_metrics['auc']:.4f}"

        f" | d+ "
        f"{val_metrics['d_pos']:.4f}"

        f" | d- "
        f"{val_metrics['d_neg']:.4f}"

        f" | gap "
        f"{val_metrics['gap']:.4f}"

        f" | d' "
        f"{val_metrics['d_prime']:.4f}"

        f" | "
        f"{train_metrics['seconds']:.1f}s"
    )

SIAMESE CNN + TRANSFORMER k=7 — EXTENDED SMOKE TEST
Epoch 04/6 | train loss 0.3701 | train AUC 0.6330 | val loss 0.4069 | val AUC 0.6295 | d+ 0.4037 | d- 0.5055 | gap 0.1018 | d' 0.4636 | 169.8s
Epoch 05/6 | train loss 0.3665 | train AUC 0.6433 | val loss 0.3851 | val AUC 0.6292 | d+ 0.4989 | d- 0.6086 | gap 0.1097 | d' 0.4679 | 145.2s
Epoch 06/6 | train loss 0.3656 | train AUC 0.6451 | val loss 0.4074 | val AUC 0.6313 | d+ 0.3944 | d- 0.4908 | gap 0.0963 | d' 0.4735 | 135.0s
